# 04 — Visualizations

Four key visualizations for the project report and README:

1. **Generation Trajectories** — animated noise-to-image evolution for DDPM, DDIM, and CFM
2. **Latent Space Interpolations** — smooth transitions between generated images
3. **Conditional Generation** — per-class samples from Elora's class-conditional CFM
4. **2D Vector Fields on Toy Data** — learned velocity fields on moons, circles, spirals

**Run on Colab with GPU** (needs trained checkpoints on Drive).

In [ ]:
import math
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, Image as IPImage
import numpy as np
from dataclasses import dataclass

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

try:
    from torchdiffeq import odeint
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'torchdiffeq'])
    from torchdiffeq import odeint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/flow-matching'
DDPM_CKPT = f'{DRIVE_DIR}/runs/ddpm/checkpoints/last.pt'
CFM_CKPT  = f'{DRIVE_DIR}/runs/cfm/checkpoints/last.pt'
ELORA_CKPT = f'{DRIVE_DIR}/runs/cfm/checkpoints/elora_epoch_190.pt'

SAVE_DIR = f'{DRIVE_DIR}/visualizations'
os.makedirs(SAVE_DIR, exist_ok=True)

---
## Model Definitions

All three architectures needed to load checkpoints.

In [ ]:
# ── DDPM config + noise scheduler ──

@dataclass
class DDPMConfig:
    image_size: int = 32
    channels: int = 3
    T: int = 1000
    schedule: str = 'cosine'
    beta_start: float = 1e-4
    beta_end: float = 0.02
    model_channels: int = 128
    num_heads: int = 4
    dropout: float = 0.1
    num_groups: int = 32
    batch_size: int = 128
    lr: float = 2e-4
    num_epochs: int = 500
    grad_clip: float = 1.0
    ema_decay: float = 0.9999
    clip_denoised: bool = True

ddpm_cfg = DDPMConfig()


def cosine_schedule(T, s=0.008):
    steps = T + 1
    t = torch.linspace(0, T, steps) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    alphas_cumprod = f / f[0]
    betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
    return betas.clamp(0, 0.999)


class NoiseScheduler:
    def __init__(self, cfg):
        self.T = cfg.T
        betas = cosine_schedule(cfg.T)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
        self.betas = betas
        self.alphas_cumprod = alphas_cumprod
        self.sqrt_alphas_cumprod = alphas_cumprod.sqrt()
        self.sqrt_one_minus_ac = (1 - alphas_cumprod).sqrt()
        self.posterior_mean_coeff1 = (alphas_cumprod_prev.sqrt() * betas) / (1 - alphas_cumprod)
        self.posterior_mean_coeff2 = (alphas.sqrt() * (1 - alphas_cumprod_prev)) / (1 - alphas_cumprod)
        self.posterior_variance = betas * (1 - alphas_cumprod_prev) / (1 - alphas_cumprod)

In [ ]:
# ── Shared UNet (DDPM + our unconditional CFM) ──

class TimeEmbedding(nn.Module):
    def __init__(self, base_dim):
        super().__init__()
        self.base_dim = base_dim
        self.mlp = nn.Sequential(nn.Linear(base_dim, base_dim*4), nn.SiLU(), nn.Linear(base_dim*4, base_dim*4))
    def forward(self, t):
        half = self.base_dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        emb = t[:, None].float() * freqs[None]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return self.mlp(emb)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_groups=32, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(num_groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch * 2))
        self.norm2 = nn.GroupNorm(num_groups, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act = nn.SiLU()
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        scale, shift = self.time_mlp(t_emb).chunk(2, dim=-1)
        h = h * (scale[:, :, None, None] + 1) + shift[:, :, None, None]
        h = self.dropout(self.conv2(self.act(self.norm2(h))))
        return h + self.shortcut(x)

class SelfAttention(nn.Module):
    def __init__(self, channels, num_heads=4, num_groups=32):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.norm = nn.GroupNorm(num_groups, channels)
        self.to_qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.to_out = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.to_qkv(h).reshape(B, 3, self.num_heads, self.head_dim, H*W).permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = F.scaled_dot_product_attention(q, k, v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        return x + self.to_out(attn)

class Downsample(nn.Module):
    def __init__(self, ch): super().__init__(); self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, ch): super().__init__(); self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x): return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))

class UNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ch, t_dim, ng, nh, do = cfg.model_channels, cfg.model_channels*4, cfg.num_groups, cfg.num_heads, cfg.dropout
        self.time_emb = TimeEmbedding(ch)
        self.init_conv = nn.Conv2d(3, ch, 3, padding=1)
        self.enc1_a = ResBlock(ch, ch, t_dim, ng, do); self.enc1_b = ResBlock(ch, ch, t_dim, ng, do); self.down1 = Downsample(ch)
        self.enc2_a = ResBlock(ch, ch*2, t_dim, ng, do); self.enc2_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.attn2 = SelfAttention(ch*2, nh, ng); self.down2 = Downsample(ch*2)
        self.enc3_a = ResBlock(ch*2, ch*2, t_dim, ng, do); self.enc3_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.attn3 = SelfAttention(ch*2, nh, ng); self.down3 = Downsample(ch*2)
        self.mid1 = ResBlock(ch*2, ch*2, t_dim, ng, do); self.mid_attn = SelfAttention(ch*2, nh, ng); self.mid2 = ResBlock(ch*2, ch*2, t_dim, ng, do)
        self.up3 = Upsample(ch*2); self.dec3_a = ResBlock(ch*4, ch*2, t_dim, ng, do); self.dec3_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.dattn3 = SelfAttention(ch*2, nh, ng)
        self.up2 = Upsample(ch*2); self.dec2_a = ResBlock(ch*4, ch*2, t_dim, ng, do); self.dec2_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.dattn2 = SelfAttention(ch*2, nh, ng)
        self.up1 = Upsample(ch*2); self.dec1_a = ResBlock(ch*3, ch, t_dim, ng, do); self.dec1_b = ResBlock(ch, ch, t_dim, ng, do)
        self.out_norm = nn.GroupNorm(ng, ch); self.out_conv = nn.Conv2d(ch, 3, 1)
    def forward(self, x, t):
        t_emb = self.time_emb(t)
        h = self.init_conv(x)
        h = self.enc1_b(self.enc1_a(h, t_emb), t_emb); s1 = h
        h = self.attn2(self.enc2_b(self.enc2_a(self.down1(h), t_emb), t_emb)); s2 = h
        h = self.attn3(self.enc3_b(self.enc3_a(self.down2(h), t_emb), t_emb)); s3 = h
        h = self.down3(h); h = self.mid1(h, t_emb); h = self.mid_attn(h); h = self.mid2(h, t_emb)
        h = self.dattn3(self.dec3_b(self.dec3_a(torch.cat([self.up3(h), s3], 1), t_emb), t_emb))
        h = self.dattn2(self.dec2_b(self.dec2_a(torch.cat([self.up2(h), s2], 1), t_emb), t_emb))
        h = self.dec1_b(self.dec1_a(torch.cat([self.up1(h), s1], 1), t_emb), t_emb)
        return self.out_conv(F.silu(self.out_norm(h)))

In [ ]:
# ── Elora's conditional SimpleFlowUNet ──

def sinusoidal_time_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / max(half - 1, 1))
    angles = t[:, None] * freqs[None, :] * 2 * math.pi
    emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb

class SelfAttention2d(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.q = nn.Conv2d(ch, ch, 1); self.k = nn.Conv2d(ch, ch, 1)
        self.v = nn.Conv2d(ch, ch, 1); self.proj = nn.Conv2d(ch, ch, 1)
    def forward(self, x):
        b, c, h, w = x.shape
        h_ = self.norm(x)
        q = self.q(h_).reshape(b, c, h*w).permute(0, 2, 1)
        k = self.k(h_).reshape(b, c, h*w)
        attn = torch.softmax(torch.bmm(q, k) * (c ** -0.5), dim=-1)
        v = self.v(h_).reshape(b, c, h*w).permute(0, 2, 1)
        out = torch.bmm(attn, v).permute(0, 2, 1).reshape(b, c, h, w)
        return x + self.proj(out)

class TimeResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.y_proj = nn.Linear(t_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb, y_emb):
        h = self.conv1(x)
        h = self.norm1(h)
        h = h + self.t_proj(t_emb)[:, :, None, None] + self.y_proj(y_emb)[:, :, None, None]
        h = F.silu(h)
        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)
        return h + self.skip(x)

class SimpleFlowUNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=128, t_dim=256, num_classes=10,
                 use_attn_16=True, use_attn_8=True):
        super().__init__()
        self.t_dim = t_dim; self.num_classes = num_classes; self.null_label = num_classes
        self.time_mlp = nn.Sequential(nn.Linear(t_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        self.class_emb = nn.Embedding(num_classes + 1, t_dim)
        self.enc1a = TimeResBlock(in_ch, base_ch, t_dim)
        self.enc1b = TimeResBlock(base_ch, base_ch, t_dim)
        self.down1 = nn.Conv2d(base_ch, base_ch*2, 4, stride=2, padding=1)
        self.enc2a = TimeResBlock(base_ch*2, base_ch*2, t_dim)
        self.enc2_attn1 = SelfAttention2d(base_ch*2) if use_attn_16 else nn.Identity()
        self.enc2b = TimeResBlock(base_ch*2, base_ch*2, t_dim)
        self.enc2_attn2 = SelfAttention2d(base_ch*2) if use_attn_16 else nn.Identity()
        self.down2 = nn.Conv2d(base_ch*2, base_ch*4, 4, stride=2, padding=1)
        self.enc3a = TimeResBlock(base_ch*4, base_ch*4, t_dim)
        self.enc3_attn = SelfAttention2d(base_ch*4) if use_attn_8 else nn.Identity()
        self.enc3b = TimeResBlock(base_ch*4, base_ch*4, t_dim)
        self.mid1 = TimeResBlock(base_ch*4, base_ch*4, t_dim)
        self.mid_attn = SelfAttention2d(base_ch*4) if use_attn_8 else nn.Identity()
        self.mid2 = TimeResBlock(base_ch*4, base_ch*4, t_dim)
        self.dec3a = TimeResBlock(base_ch*8, base_ch*4, t_dim)
        self.dec3b = TimeResBlock(base_ch*4, base_ch*4, t_dim)
        self.dec3_attn = SelfAttention2d(base_ch*4) if use_attn_8 else nn.Identity()
        self.dec2a = TimeResBlock(base_ch*6, base_ch*2, t_dim)
        self.dec2_attn1 = SelfAttention2d(base_ch*2) if use_attn_16 else nn.Identity()
        self.dec2b = TimeResBlock(base_ch*2, base_ch*2, t_dim)
        self.dec2_attn2 = SelfAttention2d(base_ch*2) if use_attn_16 else nn.Identity()
        self.dec1a = TimeResBlock(base_ch*3, base_ch, t_dim)
        self.dec1b = TimeResBlock(base_ch, base_ch, t_dim)
        self.out = nn.Conv2d(base_ch, in_ch, 3, padding=1)
    def forward(self, x, t, y=None):
        if y is None:
            y = torch.full((x.size(0),), self.null_label, device=x.device, dtype=torch.long)
        t_emb = self.time_mlp(sinusoidal_time_embedding(t, self.t_dim))
        y_emb = self.class_emb(y.long())
        s1 = self.enc1b(self.enc1a(x, t_emb, y_emb), t_emb, y_emb)
        h = self.down1(s1)
        s2 = self.enc2_attn2(self.enc2b(self.enc2_attn1(self.enc2a(h, t_emb, y_emb)), t_emb, y_emb))
        h = self.down2(s2)
        s3 = self.enc3b(self.enc3_attn(self.enc3a(h, t_emb, y_emb)), t_emb, y_emb)
        h = self.mid1(s3, t_emb, y_emb); h = self.mid_attn(h); h = self.mid2(h, t_emb, y_emb)
        h = self.dec3_attn(self.dec3b(self.dec3a(torch.cat([h, s3], 1), t_emb, y_emb), t_emb, y_emb))
        h = F.interpolate(h, scale_factor=2, mode='nearest')
        h = self.dec2_attn2(self.dec2b(self.dec2_attn1(self.dec2a(torch.cat([h, s2], 1), t_emb, y_emb)), t_emb, y_emb))
        h = F.interpolate(h, scale_factor=2, mode='nearest')
        h = self.dec1b(self.dec1a(torch.cat([h, s1], 1), t_emb, y_emb), t_emb, y_emb)
        return self.out(h)

---
## Load Models

In [ ]:
@dataclass
class CFMConfig:
    image_size: int = 32
    channels: int = 3
    model_channels: int = 128
    num_heads: int = 4
    dropout: float = 0.1
    num_groups: int = 32
    sigma_min: float = 0.0
    batch_size: int = 128
    lr: float = 2e-4
    num_epochs: int = 500
    grad_clip: float = 1.0
    ema_decay: float = 0.9999
    clip_denoised: bool = True

cfm_cfg = CFMConfig()

# DDPM
ddpm_ckpt = torch.load(DDPM_CKPT, map_location=device)
ddpm_model = UNet(ddpm_cfg).to(device)
for name, param in ddpm_model.named_parameters():
    param.data.copy_(ddpm_ckpt['ema_shadow'][name])
ddpm_model.eval()
scheduler = NoiseScheduler(ddpm_cfg)
print(f'DDPM loaded (epoch {ddpm_ckpt["epoch"]})')

# Our unconditional CFM
cfm_ckpt = torch.load(CFM_CKPT, map_location=device)
cfm_model = UNet(cfm_cfg).to(device)
for name, param in cfm_model.named_parameters():
    param.data.copy_(cfm_ckpt['ema_shadow'][name])
cfm_model.eval()
print(f'CFM loaded (epoch {cfm_ckpt["epoch"]})')

# Elora's conditional CFM
elora_ckpt = torch.load(ELORA_CKPT, map_location=device)
elora_cfg = elora_ckpt['config']
elora_model = SimpleFlowUNet(
    in_ch=3, base_ch=elora_cfg['base_ch'], t_dim=elora_cfg['t_dim'],
    num_classes=elora_cfg['num_classes'],
    use_attn_16=elora_cfg['use_attn_16'], use_attn_8=elora_cfg['use_attn_8'],
).to(device)
elora_model.load_state_dict(elora_ckpt['ema_state_dict'])
elora_model.eval()
print(f'Elora conditional CFM loaded (epoch {elora_ckpt["epoch"]})')

---
## Visualization 1: Generation Trajectories (Animated)

Watch noise become images in real-time:
- **DDPM** (1000 steps): slow, curved denoising path
- **DDIM** (50 steps): same model, faster deterministic sampling
- **OT-CFM** (50 NFE, midpoint): straight ODE integration

Each saved as an animated GIF.

In [ ]:
def to_img(x):
    """[-1,1] tensor -> [0,1] numpy HWC."""
    return ((x + 1) / 2).clamp(0, 1).permute(1, 2, 0).cpu().numpy()


@torch.no_grad()
def ddpm_trajectory(model, scheduler, shape, device, save_every=20):
    """DDPM reverse process, saving frames at regular intervals."""
    model.eval()
    x = torch.randn(shape, device=device)
    frames = [(999, x.clone().cpu())]
    for t in reversed(range(scheduler.T)):
        t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
        eps_hat = model(x, t_batch)
        sqrt_ac = scheduler.sqrt_alphas_cumprod[t]
        sqrt_omc = scheduler.sqrt_one_minus_ac[t]
        x0_pred = ((x - sqrt_omc * eps_hat) / sqrt_ac).clamp(-1, 1)
        coeff1 = scheduler.posterior_mean_coeff1[t]
        coeff2 = scheduler.posterior_mean_coeff2[t]
        mean = coeff1 * x0_pred + coeff2 * x
        x = mean + (scheduler.posterior_variance[t].sqrt() * torch.randn_like(x) if t > 0 else 0)
        if t % save_every == 0:
            frames.append((t, x.clone().cpu()))
    return frames


@torch.no_grad()
def ddim_trajectory(model, scheduler, shape, device, num_steps=50):
    """DDIM sampling, saving every step."""
    model.eval()
    step = scheduler.T // num_steps
    steps = list(reversed(range(0, scheduler.T, step)))
    x = torch.randn(shape, device=device)
    frames = [(steps[0], x.clone().cpu())]
    for i, t in enumerate(steps):
        t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
        t_prev = steps[i + 1] if i + 1 < len(steps) else 0
        eps_hat = model(x, t_batch)
        ac = scheduler.alphas_cumprod[t].to(device)
        ac_prev = scheduler.alphas_cumprod[t_prev].to(device) if t_prev > 0 else torch.tensor(1.0, device=device)
        x0_pred = ((x - (1 - ac).sqrt() * eps_hat) / ac.sqrt()).clamp(-1, 1)
        x = ac_prev.sqrt() * x0_pred + (1 - ac_prev).sqrt() * eps_hat
        frames.append((t_prev, x.clone().cpu()))
    return frames


@torch.no_grad()
def cfm_trajectory(model, shape, device, nfe=50, num_frames=50):
    """CFM ODE integration, saving intermediate states."""
    model.eval()
    t_eval = torch.linspace(0, 1, num_frames + 1, device=device)

    def ode_fn(t, x):
        t_batch = torch.full((shape[0],), t.item(), device=device)
        return model(x, t_batch)

    x0 = torch.randn(shape, device=device)
    nfe_per_step = 2  # midpoint
    step_size = nfe_per_step / nfe
    trajectory = odeint(ode_fn, x0, t_eval, method='midpoint',
                        options={'step_size': step_size})
    frames = []
    for i, t_val in enumerate(t_eval):
        frames.append((t_val.item(), trajectory[i].clone().cpu()))
    return frames

In [ ]:
torch.manual_seed(42)
n_samples = 4
shape = (n_samples, 3, 32, 32)

# Use the same initial noise for DDIM and DDPM (they share the same model anyway)
# CFM uses its own noise since it's a different model
print('Generating DDPM trajectory (1000 steps)...')
ddpm_frames = ddpm_trajectory(ddpm_model, scheduler, shape, device, save_every=20)

print('Generating DDIM trajectory (50 steps)...')
ddim_frames = ddim_trajectory(ddpm_model, scheduler, shape, device, num_steps=50)

print('Generating CFM trajectory (50 NFE, midpoint)...')
cfm_frames = cfm_trajectory(cfm_model, shape, device, nfe=50, num_frames=50)

print('Done!')

In [ ]:
def make_trajectory_gif(frames, title, save_path, n_samples=4, fps=15):
    """Create an animated GIF from trajectory frames."""
    fig, axes = plt.subplots(1, n_samples, figsize=(3 * n_samples, 3))
    fig.suptitle(title, fontsize=14, y=1.02)
    if n_samples == 1:
        axes = [axes]
    ims = []
    for ax in axes:
        im = ax.imshow(np.zeros((32, 32, 3)))
        ax.axis('off')
        ims.append(im)
    step_text = fig.text(0.5, -0.02, '', ha='center', fontsize=11)
    plt.tight_layout()

    def update(frame_idx):
        step_val, batch = frames[frame_idx]
        for i, im in enumerate(ims):
            im.set_data(to_img(batch[i]))
        if isinstance(step_val, float):
            step_text.set_text(f't = {step_val:.2f}')
        else:
            step_text.set_text(f't = {step_val}')
        return ims + [step_text]

    anim = FuncAnimation(fig, update, frames=len(frames), interval=1000//fps, blit=True)
    anim.save(save_path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    print(f'Saved: {save_path}')
    return anim


# Generate GIFs
anim_ddpm = make_trajectory_gif(ddpm_frames, 'DDPM (1000 steps)',
                                f'{SAVE_DIR}/trajectory_ddpm.gif', n_samples)
anim_ddim = make_trajectory_gif(ddim_frames, 'DDIM (50 steps)',
                                f'{SAVE_DIR}/trajectory_ddim.gif', n_samples)
anim_cfm = make_trajectory_gif(cfm_frames, 'OT-CFM (50 NFE, midpoint)',
                               f'{SAVE_DIR}/trajectory_cfm.gif', n_samples)

# Display inline
print('\n--- DDPM ---')
display(HTML(anim_ddpm.to_jshtml()))
print('\n--- DDIM ---')
display(HTML(anim_ddim.to_jshtml()))
print('\n--- OT-CFM ---')
display(HTML(anim_cfm.to_jshtml()))

---
## Visualization 2: Latent Space Interpolations

Interpolate between two noise vectors z_A and z_B in latent space:

> z_α = (1 - α) · z_A + α · z_B,  α ∈ [0, 1]

Then generate an image from each z_α. Smooth transitions = meaningful latent space.

We use spherical linear interpolation (slerp) for better results in high dimensions.

In [ ]:
def slerp(z1, z2, alpha):
    """Spherical linear interpolation between two tensors."""
    z1_flat = z1.flatten()
    z2_flat = z2.flatten()
    cos_omega = (z1_flat @ z2_flat) / (z1_flat.norm() * z2_flat.norm())
    cos_omega = cos_omega.clamp(-1, 1)
    omega = torch.acos(cos_omega)
    if omega.abs() < 1e-6:
        return (1 - alpha) * z1 + alpha * z2
    return (torch.sin((1 - alpha) * omega) / torch.sin(omega)) * z1 + \
           (torch.sin(alpha * omega) / torch.sin(omega)) * z2


@torch.no_grad()
def cfm_generate(model, z, device, nfe=50):
    """Generate images from specific noise vectors using CFM."""
    model.eval()
    n = z.shape[0]
    def ode_fn(t, x):
        t_batch = torch.full((n,), t.item(), device=device)
        return model(x, t_batch)
    t_span = torch.tensor([0.0, 1.0], device=device)
    x = odeint(ode_fn, z.to(device), t_span, method='midpoint',
               options={'step_size': 2 / nfe})[-1]
    return x.clamp(-1, 1)


@torch.no_grad()
def ddim_generate(model, scheduler, z, device, num_steps=50):
    """Generate images from specific noise vectors using DDIM."""
    model.eval()
    step = scheduler.T // num_steps
    steps = list(reversed(range(0, scheduler.T, step)))
    x = z.to(device)
    for i, t in enumerate(steps):
        t_batch = torch.full((x.shape[0],), t, device=device, dtype=torch.long)
        t_prev = steps[i + 1] if i + 1 < len(steps) else 0
        eps_hat = model(x, t_batch)
        ac = scheduler.alphas_cumprod[t].to(device)
        ac_prev = scheduler.alphas_cumprod[t_prev].to(device) if t_prev > 0 else torch.tensor(1.0, device=device)
        x0_pred = ((x - (1 - ac).sqrt() * eps_hat) / ac.sqrt()).clamp(-1, 1)
        x = ac_prev.sqrt() * x0_pred + (1 - ac_prev).sqrt() * eps_hat
    return x

In [ ]:
n_interp = 10  # number of interpolation steps
n_pairs = 3    # number of pairs to show
alphas = torch.linspace(0, 1, n_interp)

torch.manual_seed(123)
z_pairs = [(torch.randn(1, 3, 32, 32), torch.randn(1, 3, 32, 32)) for _ in range(n_pairs)]

# ── CFM interpolations ──
print('Generating CFM latent interpolations...')
cfm_interp_rows = []
for z_a, z_b in z_pairs:
    z_batch = torch.cat([slerp(z_a, z_b, a.item()) for a in alphas], dim=0)
    imgs = cfm_generate(cfm_model, z_batch, device, nfe=50)
    cfm_interp_rows.append(imgs.cpu())

# ── DDIM interpolations ──
print('Generating DDIM latent interpolations...')
ddim_interp_rows = []
for z_a, z_b in z_pairs:
    z_batch = torch.cat([slerp(z_a, z_b, a.item()) for a in alphas], dim=0)
    imgs = ddim_generate(ddpm_model, scheduler, z_batch, device, num_steps=50)
    ddim_interp_rows.append(imgs.cpu())

print('Done!')

In [ ]:
def plot_interpolation_grid(rows, title, save_path):
    n_pairs = len(rows)
    n_interp = rows[0].shape[0]
    fig, axes = plt.subplots(n_pairs, n_interp, figsize=(2 * n_interp, 2.2 * n_pairs))
    fig.suptitle(title, fontsize=14, y=1.02)
    for row_idx in range(n_pairs):
        for col_idx in range(n_interp):
            ax = axes[row_idx, col_idx] if n_pairs > 1 else axes[col_idx]
            ax.imshow(to_img(rows[row_idx][col_idx]))
            ax.axis('off')
            if row_idx == 0:
                alpha_val = col_idx / (n_interp - 1)
                ax.set_title(f'α={alpha_val:.1f}', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

plot_interpolation_grid(cfm_interp_rows, 'OT-CFM Latent Interpolation (slerp)',
                        f'{SAVE_DIR}/interpolation_cfm.png')
plot_interpolation_grid(ddim_interp_rows, 'DDIM Latent Interpolation (slerp)',
                        f'{SAVE_DIR}/interpolation_ddim.png')

In [ ]:
def make_interpolation_gif(rows, title, save_path, fps=8):
    """Animate each interpolation pair as a smooth transition."""
    n_pairs = len(rows)
    n_interp = rows[0].shape[0]
    fig, axes = plt.subplots(1, n_pairs, figsize=(3 * n_pairs, 3))
    fig.suptitle(title, fontsize=13, y=1.02)
    if n_pairs == 1:
        axes = [axes]
    ims = []
    for ax in axes:
        im = ax.imshow(np.zeros((32, 32, 3)))
        ax.axis('off')
        ims.append(im)
    alpha_text = fig.text(0.5, -0.02, '', ha='center', fontsize=11)
    plt.tight_layout()

    # Ping-pong: go forward then backward for smooth loop
    indices = list(range(n_interp)) + list(range(n_interp - 2, 0, -1))

    def update(frame_idx):
        idx = indices[frame_idx]
        for i, im in enumerate(ims):
            im.set_data(to_img(rows[i][idx]))
        alpha_text.set_text(f'α = {idx / (n_interp - 1):.2f}')
        return ims + [alpha_text]

    anim = FuncAnimation(fig, update, frames=len(indices), interval=1000//fps, blit=True)
    anim.save(save_path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    print(f'Saved: {save_path}')
    return anim

anim_interp_cfm = make_interpolation_gif(cfm_interp_rows, 'OT-CFM Interpolation',
                                          f'{SAVE_DIR}/interpolation_cfm.gif')
anim_interp_ddim = make_interpolation_gif(ddim_interp_rows, 'DDIM Interpolation',
                                           f'{SAVE_DIR}/interpolation_ddim.gif')

display(HTML(anim_interp_cfm.to_jshtml()))
display(HTML(anim_interp_ddim.to_jshtml()))

---
## Visualization 3: Conditional Generation

Using Elora's class-conditional CFM, generate a grid where each row is a CIFAR-10 class.

Also show the effect of classifier-free guidance (cfg_scale) on sample quality.

In [ ]:
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']


@torch.no_grad()
def conditional_sample(model, labels, device, nfe=100, cfg_scale=1.0, seed=42):
    """Sample from Elora's conditional CFM with optional CFG."""
    model.eval()
    n = labels.shape[0]
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    x0 = torch.randn(n, 3, 32, 32, device=device, generator=gen)
    null_labels = torch.full_like(labels, model.null_label)

    def ode_fn(t, x):
        t_batch = torch.full((n,), t.item(), device=device)
        v_cond = model(x, t_batch, labels)
        if cfg_scale != 1.0:
            v_uncond = model(x, t_batch, null_labels)
            return v_uncond + cfg_scale * (v_cond - v_uncond)
        return v_cond

    t_span = torch.tensor([0.0, 1.0], device=device)
    x = odeint(ode_fn, x0, t_span, method='midpoint',
               options={'step_size': 2 / nfe})[-1]
    return x.clamp(-1, 1)

In [ ]:
# Generate 8 samples per class (10 classes x 8 = 80 images)
samples_per_class = 8
labels = torch.arange(10, device=device).repeat_interleave(samples_per_class)

print('Generating class-conditional samples (cfg_scale=1.0)...')
cond_samples_1 = conditional_sample(elora_model, labels, device, nfe=100, cfg_scale=1.0, seed=42)

print('Generating class-conditional samples (cfg_scale=2.0)...')
cond_samples_2 = conditional_sample(elora_model, labels, device, nfe=100, cfg_scale=2.0, seed=42)

print('Done!')

In [ ]:
def plot_class_grid(samples, title, save_path, samples_per_class=8):
    """Plot a grid: rows = classes, cols = samples."""
    n_classes = 10
    fig, axes = plt.subplots(n_classes, samples_per_class,
                             figsize=(1.8 * samples_per_class, 1.8 * n_classes))
    fig.suptitle(title, fontsize=14, y=1.01)
    for cls_idx in range(n_classes):
        axes[cls_idx, 0].set_ylabel(CLASSES[cls_idx], fontsize=10, rotation=0,
                                     labelpad=60, va='center')
        for s_idx in range(samples_per_class):
            img_idx = cls_idx * samples_per_class + s_idx
            axes[cls_idx, s_idx].imshow(to_img(samples[img_idx]))
            axes[cls_idx, s_idx].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

plot_class_grid(cond_samples_1.cpu(), 'Conditional CFM — cfg_scale = 1.0 (no guidance)',
                f'{SAVE_DIR}/conditional_cfg1.png')
plot_class_grid(cond_samples_2.cpu(), 'Conditional CFM — cfg_scale = 2.0 (with guidance)',
                f'{SAVE_DIR}/conditional_cfg2.png')

In [ ]:
# Side-by-side CFG comparison for a single class
cfg_scales = [1.0, 1.5, 2.0, 3.0]
demo_class = 7  # horse
n_demo = 8
demo_labels = torch.full((n_demo,), demo_class, device=device, dtype=torch.long)

print(f'Generating "{CLASSES[demo_class]}" at different CFG scales...')
cfg_samples = {}
for scale in cfg_scales:
    cfg_samples[scale] = conditional_sample(
        elora_model, demo_labels, device, nfe=100, cfg_scale=scale, seed=0
    ).cpu()

fig, axes = plt.subplots(len(cfg_scales), n_demo,
                         figsize=(1.8 * n_demo, 1.8 * len(cfg_scales)))
fig.suptitle(f'Classifier-Free Guidance Effect — "{CLASSES[demo_class]}"', fontsize=14, y=1.01)
for row, scale in enumerate(cfg_scales):
    axes[row, 0].set_ylabel(f'cfg={scale}', fontsize=10, rotation=0, labelpad=45, va='center')
    for col in range(n_demo):
        axes[row, col].imshow(to_img(cfg_samples[scale][col]))
        axes[row, col].axis('off')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/conditional_cfg_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {SAVE_DIR}/conditional_cfg_comparison.png')

---
## Visualization 4: 2D Vector Fields on Toy Data

Train a tiny flow matching model on 2D synthetic datasets (moons, circles, spirals).

Then visualize the learned velocity field as arrows — showing how the model transports noise into data.

In [ ]:
from sklearn.datasets import make_moons, make_circles


def make_spirals(n_samples=1000, noise=0.05):
    """Two interleaving spirals."""
    n = n_samples // 2
    theta = torch.linspace(0, 4 * np.pi, n)
    r = torch.linspace(0.2, 2, n)
    x1 = torch.stack([r * theta.cos(), r * theta.sin()], dim=1)
    x2 = torch.stack([-r * theta.cos(), -r * theta.sin()], dim=1)
    data = torch.cat([x1, x2], dim=0)
    data += noise * torch.randn_like(data)
    return data.numpy()


class TinyFlowNet(nn.Module):
    """Small MLP for 2D flow matching."""
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.SiLU(),   # input: [x, y, t]
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),               # output: [vx, vy]
        )
    def forward(self, x, t):
        if t.dim() == 0:
            t = t.expand(x.shape[0])
        return self.net(torch.cat([x, t.unsqueeze(-1)], dim=-1))


def train_toy_cfm(data, n_steps=5000, lr=1e-3, batch_size=256):
    """Train a 2D flow matching model."""
    x1_all = torch.tensor(data, dtype=torch.float32)
    model = TinyFlowNet()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for step in range(n_steps):
        idx = torch.randint(0, len(x1_all), (batch_size,))
        x1 = x1_all[idx]
        x0 = torch.randn_like(x1)
        t = torch.rand(batch_size)
        x_t = (1 - t.unsqueeze(-1)) * x0 + t.unsqueeze(-1) * x1
        target = x1 - x0
        pred = model(x_t, t)
        loss = F.mse_loss(pred, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    return model


@torch.no_grad()
def sample_toy(model, n=1000, steps=100):
    """Euler sampling for 2D flow matching."""
    x = torch.randn(n, 2)
    dt = 1.0 / steps
    for i in range(steps):
        t = torch.full((n,), i * dt)
        x = x + dt * model(x, t)
    return x.numpy()

In [ ]:
# Generate datasets
np.random.seed(42)
torch.manual_seed(42)

datasets_2d = {
    'Moons': make_moons(n_samples=2000, noise=0.05)[0],
    'Circles': make_circles(n_samples=2000, noise=0.03, factor=0.5)[0],
    'Spirals': make_spirals(n_samples=2000, noise=0.05),
}

# Train models
toy_models = {}
for name, data in datasets_2d.items():
    print(f'Training on {name}...')
    toy_models[name] = train_toy_cfm(data, n_steps=5000)

print('All 2D models trained!')

In [ ]:
@torch.no_grad()
def plot_vector_field(model, data, title, ax, t_val=0.5, grid_res=25):
    """Plot the learned velocity field at a given time t."""
    # Data extent
    margin = 0.5
    x_min, x_max = data[:, 0].min() - margin, data[:, 0].max() + margin
    y_min, y_max = data[:, 1].min() - margin, data[:, 1].max() + margin

    # Grid
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_res),
                         np.linspace(y_min, y_max, grid_res))
    grid = torch.tensor(np.stack([xx.ravel(), yy.ravel()], axis=1), dtype=torch.float32)
    t = torch.full((grid.shape[0],), t_val)
    vel = model(grid, t).numpy()

    # Plot
    ax.scatter(data[:, 0], data[:, 1], c='steelblue', s=3, alpha=0.3, label='data')
    ax.quiver(xx.ravel(), yy.ravel(), vel[:, 0], vel[:, 1],
              color='tomato', alpha=0.7, scale=30, width=0.004)
    ax.set_title(f'{title} (t={t_val})', fontsize=12)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)


# ── Vector fields at multiple time steps ──
time_steps = [0.0, 0.25, 0.5, 0.75, 1.0]
dataset_names = list(datasets_2d.keys())

fig, axes = plt.subplots(len(dataset_names), len(time_steps),
                         figsize=(4 * len(time_steps), 4 * len(dataset_names)))
fig.suptitle('Learned Velocity Fields at Different Times', fontsize=15, y=1.01)

for row, name in enumerate(dataset_names):
    for col, t_val in enumerate(time_steps):
        plot_vector_field(toy_models[name], datasets_2d[name], name, axes[row, col], t_val=t_val)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/vector_fields_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {SAVE_DIR}/vector_fields_2d.png')

In [ ]:
# ── Generated samples vs real data ──
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('2D Flow Matching: Real Data vs Generated Samples', fontsize=14)

for col, name in enumerate(dataset_names):
    data = datasets_2d[name]
    generated = sample_toy(toy_models[name], n=2000, steps=200)

    axes[0, col].scatter(data[:, 0], data[:, 1], c='steelblue', s=5, alpha=0.5)
    axes[0, col].set_title(f'{name} — Real', fontsize=12)
    axes[0, col].set_aspect('equal')
    axes[0, col].grid(True, alpha=0.2)

    axes[1, col].scatter(generated[:, 0], generated[:, 1], c='tomato', s=5, alpha=0.5)
    axes[1, col].set_title(f'{name} — Generated (200 steps)', fontsize=12)
    axes[1, col].set_aspect('equal')
    axes[1, col].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/toy_2d_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {SAVE_DIR}/toy_2d_samples.png')

In [ ]:
# ── Animated: particles + vector field evolving together ──
TOY_STEPS = 200  # sampling steps (increase for better convergence)

@torch.no_grad()
def toy_trajectory_with_field(model, data, n=500, steps=TOY_STEPS, grid_res=20):
    """Collect particle positions AND velocity field at each frame."""
    x = torch.randn(n, 2)
    dt = 1.0 / steps

    margin = 0.5
    x_min, x_max = data[:, 0].min() - margin, data[:, 0].max() + margin
    y_min, y_max = data[:, 1].min() - margin, data[:, 1].max() + margin
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_res),
                         np.linspace(y_min, y_max, grid_res))
    grid = torch.tensor(np.stack([xx.ravel(), yy.ravel()], axis=1), dtype=torch.float32)

    frames = []
    for i in range(steps + 1):
        t_val = i * dt
        t_grid = torch.full((grid.shape[0],), t_val)
        vel = model(grid, t_grid).numpy()
        frames.append((t_val, x.numpy().copy(), vel.copy()))
        if i < steps:
            t_batch = torch.full((n,), t_val)
            x = x + dt * model(x, t_batch)

    return frames, xx, yy


torch.manual_seed(0)
dataset_names = list(datasets_2d.keys())

# Pre-compute all frames
toy_field_data = {}
toy_grids = {}
for name in dataset_names:
    print(f'Generating trajectory + field for {name}...')
    frames, xx, yy = toy_trajectory_with_field(
        toy_models[name], datasets_2d[name], n=500, steps=TOY_STEPS, grid_res=20
    )
    toy_field_data[name] = frames
    toy_grids[name] = (xx, yy)

# Use ALL frames, subsample evenly to ~80 frames for a smooth GIF
total_frames = len(toy_field_data[dataset_names[0]])
target_gif_frames = 80
skip = max(1, total_frames // target_gif_frames)
frame_indices = list(range(0, total_frames, skip))
if frame_indices[-1] != total_frames - 1:
    frame_indices.append(total_frames - 1)  # always include the last frame

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('2D Flow Matching: Particles + Velocity Field', fontsize=14, y=1.02)

# Initialize plots
scatters = []
quivers = []
for col, name in enumerate(dataset_names):
    data = datasets_2d[name]
    margin = 0.5
    ax = axes[col]
    ax.set_xlim(data[:, 0].min() - margin, data[:, 0].max() + margin)
    ax.set_ylim(data[:, 1].min() - margin, data[:, 1].max() + margin)
    ax.set_aspect('equal')
    ax.set_title(name, fontsize=12)
    ax.grid(True, alpha=0.2)

    # Target data in background
    ax.scatter(data[:, 0], data[:, 1], c='steelblue', s=3, alpha=0.15, zorder=1)

    sc = ax.scatter([], [], c='tomato', s=5, alpha=0.6, zorder=3)
    scatters.append(sc)

    xx, yy = toy_grids[name]
    q = ax.quiver(xx.ravel(), yy.ravel(),
                  np.zeros(xx.size), np.zeros(xx.size),
                  color='gray', alpha=0.5, scale=35, width=0.004, zorder=2)
    quivers.append(q)

step_text = fig.text(0.5, -0.01, '', ha='center', fontsize=11)
plt.tight_layout()


def update(fi):
    idx = frame_indices[fi]
    for col, name in enumerate(dataset_names):
        t_val, pts, vel = toy_field_data[name][idx]
        scatters[col].set_offsets(pts)
        quivers[col].set_UVC(vel[:, 0], vel[:, 1])
    step_text.set_text(f't = {toy_field_data[dataset_names[0]][idx][0]:.2f}')
    return scatters + quivers + [step_text]


anim_toy = FuncAnimation(fig, update, frames=len(frame_indices), interval=80, blit=True)
anim_toy.save(f'{SAVE_DIR}/toy_2d_trajectory.gif', writer=PillowWriter(fps=15))
plt.close(fig)
print(f'Saved: {SAVE_DIR}/toy_2d_trajectory.gif')
HTML(anim_toy.to_jshtml())

---
## Saved Files

All visualizations saved to `flow-matching/visualizations/` on Drive:

| File | Description |
|------|-------------|
| `trajectory_ddpm.gif` | DDPM denoising animation (1000 steps) |
| `trajectory_ddim.gif` | DDIM denoising animation (50 steps) |
| `trajectory_cfm.gif` | OT-CFM generation animation (50 NFE) |
| `interpolation_cfm.png` | CFM latent space interpolation grid |
| `interpolation_ddim.png` | DDIM latent space interpolation grid |
| `interpolation_cfm.gif` | CFM interpolation animation |
| `interpolation_ddim.gif` | DDIM interpolation animation |
| `conditional_cfg1.png` | Per-class samples (cfg_scale=1.0) |
| `conditional_cfg2.png` | Per-class samples (cfg_scale=2.0) |
| `conditional_cfg_comparison.png` | CFG scale comparison |
| `vector_fields_2d.png` | Learned velocity fields on toy data |
| `toy_2d_samples.png` | Real vs generated 2D distributions |
| `toy_2d_trajectory.gif` | 2D noise-to-data animation |